# Information Retrieval Classical Demo

This notebook demonstrates classical information retrieval techniques including document preprocessing, inverted indexing, boolean search, and ranked retrieval algorithms.

In [4]:
# Import Required Libraries
import re
import time
import string
from collections import defaultdict, Counter
import math
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

True

## Load and Explore Corpus

We use a small corpus of 10 documents about animals for demonstration.

In [2]:
# Define the corpus
corpus = [
    "The quick brown fox jumps over the lazy dog.",
    "A brown fox is quick and agile.",
    "The lazy dog sleeps all day.",
    "Jumping over obstacles is fun.",
    "Dogs are loyal animals.",
    "Foxes are cunning hunters.",
    "Animals live in various habitats.",
    "Birds fly in the sky.",
    "Fish swim in the water.",
    "Lions are kings of the jungle."
]

# Corpus statistics
num_docs = len(corpus)
avg_length = sum(len(doc.split()) for doc in corpus) / num_docs

print(f"Number of documents: {num_docs}")
print(f"Average document length: {avg_length:.2f} words")
print("\nSample documents:")
for i, doc in enumerate(corpus[:3]):
    print(f"Doc {i+1}: {doc}")

Number of documents: 10
Average document length: 5.60 words

Sample documents:
Doc 1: The quick brown fox jumps over the lazy dog.
Doc 2: A brown fox is quick and agile.
Doc 3: The lazy dog sleeps all day.


## Document Preprocessing

Preprocessing includes lowercasing, punctuation removal, tokenization, stopword removal, and stemming.

In [6]:
def preprocess(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
    # Tokenize (simple split for demo)
    tokens = text.split()
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words]
    # Stem
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens

# Preprocess the corpus
processed_corpus = [preprocess(doc) for doc in corpus]

print("Original:", corpus[0])
print("Processed:", processed_corpus[0])

Original: The quick brown fox jumps over the lazy dog.
Processed: ['quick', 'brown', 'fox', 'jump', 'lazi', 'dog']


## Build Inverted Index

The inverted index maps terms to lists of (doc_id, position) pairs.

In [7]:
# Build inverted index
inverted_index = defaultdict(list)
for doc_id, tokens in enumerate(processed_corpus):
    for pos, term in enumerate(tokens):
        inverted_index[term].append((doc_id, pos))

# Sort postings
for term in inverted_index:
    inverted_index[term].sort()

print("Sample inverted index entries:")
for term in sorted(list(inverted_index.keys())[:5]):
    print(f"{term}: {inverted_index[term]}")

Sample inverted index entries:
brown: [(0, 1), (1, 0)]
fox: [(0, 2), (1, 1), (5, 0)]
jump: [(0, 3), (3, 0)]
lazi: [(0, 4), (2, 0)]
quick: [(0, 0), (1, 2)]


## Boolean Query Search

Implement single-term boolean search.

In [8]:
def boolean_search_single(term):
    processed = preprocess(term)
    if not processed:
        return set()
    term = processed[0]
    if term in inverted_index:
        docs = set(doc_id for doc_id, _ in inverted_index[term])
        return docs
    return set()

# Example
start = time.time()
results = boolean_search_single("fox")
end = time.time()
print(f"Query 'fox' results: {results}")
print(f"Time: {end - start:.6f} seconds")
for doc_id in sorted(results):
    print(f"Doc {doc_id+1}: {corpus[doc_id]}")

Query 'fox' results: {0, 1, 5}
Time: 0.001079 seconds
Doc 1: The quick brown fox jumps over the lazy dog.
Doc 2: A brown fox is quick and agile.
Doc 6: Foxes are cunning hunters.


## Multi-term Boolean Search

Implement AND and OR operations.

In [9]:
def boolean_search_and(terms):
    processed_terms = [preprocess(t)[0] for t in terms if preprocess(t)]
    if not processed_terms:
        return set()
    result = set(range(len(corpus)))
    for term in processed_terms:
        if term in inverted_index:
            docs = set(doc_id for doc_id, _ in inverted_index[term])
            result = result & docs
        else:
            return set()
    return result

def boolean_search_or(terms):
    processed_terms = [preprocess(t)[0] for t in terms if preprocess(t)]
    if not processed_terms:
        return set()
    result = set()
    for term in processed_terms:
        if term in inverted_index:
            docs = set(doc_id for doc_id, _ in inverted_index[term])
            result = result | docs
    return result

# Example AND
start = time.time()
results_and = boolean_search_and(["fox", "brown"])
end = time.time()
print(f"Query 'fox AND brown' results: {results_and}")
print(f"Time: {end - start:.6f} seconds")
for doc_id in sorted(results_and):
    print(f"Doc {doc_id+1}: {corpus[doc_id]}")

# Example OR
start = time.time()
results_or = boolean_search_or(["fox", "dog"])
end = time.time()
print(f"\nQuery 'fox OR dog' results: {results_or}")
print(f"Time: {end - start:.6f} seconds")
for doc_id in sorted(results_or):
    print(f"Doc {doc_id+1}: {corpus[doc_id]}")

Query 'fox AND brown' results: {0, 1}
Time: 0.001933 seconds
Doc 1: The quick brown fox jumps over the lazy dog.
Doc 2: A brown fox is quick and agile.

Query 'fox OR dog' results: {0, 1, 2, 4, 5}
Time: 0.001521 seconds
Doc 1: The quick brown fox jumps over the lazy dog.
Doc 2: A brown fox is quick and agile.
Doc 3: The lazy dog sleeps all day.
Doc 5: Dogs are loyal animals.
Doc 6: Foxes are cunning hunters.


## Proximity-based Boolean Search

Implement proximity search for terms within k positions.

In [10]:
def proximity_search(term1, term2, k):
    t1 = preprocess(term1)[0]
    t2 = preprocess(term2)[0]
    if t1 not in inverted_index or t2 not in inverted_index:
        return set()
    postings1 = inverted_index[t1]
    postings2 = inverted_index[t2]
    result = set()
    i, j = 0, 0
    while i < len(postings1) and j < len(postings2):
        doc1, pos1 = postings1[i]
        doc2, pos2 = postings2[j]
        if doc1 == doc2:
            if abs(pos1 - pos2) <= k:
                result.add(doc1)
            if pos1 < pos2:
                i += 1
            else:
                j += 1
        elif doc1 < doc2:
            i += 1
        else:
            j += 1
    return result

# Example
start = time.time()
results_prox = proximity_search("brown", "fox", 2)
end = time.time()
print(f"Query 'brown NEAR(2) fox' results: {results_prox}")
print(f"Time: {end - start:.6f} seconds")
for doc_id in sorted(results_prox):
    print(f"Doc {doc_id+1}: {corpus[doc_id]}")

# Compare with AND
results_and_comp = boolean_search_and(["brown", "fox"])
print(f"\nAND results: {results_and_comp}")

Query 'brown NEAR(2) fox' results: {0, 1}
Time: 0.001257 seconds
Doc 1: The quick brown fox jumps over the lazy dog.
Doc 2: A brown fox is quick and agile.

AND results: {0, 1}


## TF-IDF Ranked Retrieval

Implement TF-IDF scoring for ranked retrieval.

In [11]:
# Compute DF and IDF
df = {term: len(set(doc for doc, _ in postings)) for term, postings in inverted_index.items()}
idf = {term: math.log(num_docs / df[term]) for term in df}

def tfidf_search(query):
    processed_query = preprocess(query)
    if not processed_query:
        return []
    scores = defaultdict(float)
    for term in processed_query:
        if term in inverted_index:
            idf_val = idf[term]
            postings = inverted_index[term]
            doc_tf = defaultdict(int)
            for doc_id, _ in postings:
                doc_tf[doc_id] += 1
            for doc_id, tf in doc_tf.items():
                scores[doc_id] += tf * idf_val
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return ranked

# Example
start = time.time()
results_tfidf = tfidf_search("brown fox")
end = time.time()
print(f"Query 'brown fox' TF-IDF results:")
print(f"Time: {end - start:.6f} seconds")
for doc_id, score in results_tfidf[:5]:
    print(f"Doc {doc_id+1}: Score {score:.4f} - {corpus[doc_id]}")

Query 'brown fox' TF-IDF results:
Time: 0.001398 seconds
Doc 1: Score 2.8134 - The quick brown fox jumps over the lazy dog.
Doc 2: Score 2.8134 - A brown fox is quick and agile.
Doc 6: Score 1.2040 - Foxes are cunning hunters.


## BM25 Ranked Retrieval

Implement BM25 scoring.

In [12]:
# Compute average document length
avg_doc_len = sum(len(tokens) for tokens in processed_corpus) / num_docs

def bm25_search(query, k1=1.5, b=0.75):
    processed_query = preprocess(query)
    if not processed_query:
        return []
    scores = defaultdict(float)
    for term in processed_query:
        if term in inverted_index:
            idf_val = idf[term]
            postings = inverted_index[term]
            doc_tf = defaultdict(int)
            for doc_id, _ in postings:
                doc_tf[doc_id] += 1
            for doc_id, tf in doc_tf.items():
                doc_len = len(processed_corpus[doc_id])
                score = idf_val * (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * doc_len / avg_doc_len))
                scores[doc_id] += score
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return ranked

# Example
start = time.time()
results_bm25 = bm25_search("brown fox")
end = time.time()
print(f"Query 'brown fox' BM25 results:")
print(f"Time: {end - start:.6f} seconds")
for doc_id, score in results_bm25[:5]:
    print(f"Doc {doc_id+1}: Score {score:.4f} - {corpus[doc_id]}")

# Compare with TF-IDF
print(f"\nTF-IDF top: Doc {results_tfidf[0][0]+1} Score {results_tfidf[0][1]:.4f}")
print(f"BM25 top: Doc {results_bm25[0][0]+1} Score {results_bm25[0][1]:.4f}")

Query 'brown fox' BM25 results:
Time: 0.001322 seconds
Doc 2: Score 2.6794 - A brown fox is quick and agile.
Doc 1: Score 2.1642 - The quick brown fox jumps over the lazy dog.
Doc 6: Score 1.3016 - Foxes are cunning hunters.

TF-IDF top: Doc 1 Score 2.8134
BM25 top: Doc 2 Score 2.6794


## Performance Comparison and Analysis

Compare execution times and outputs for different algorithms.

In [ ]:
# Performance comparison
queries = ["fox", "brown fox", "animal"]

algorithms = {
    "Boolean AND": lambda q: boolean_search_and(q.split()),
    "TF-IDF": tfidf_search,
    "BM25": bm25_search
}

for query in queries:
    print(f"\nQuery: '{query}'")
    for name, func in algorithms.items():
        start = time.time()
        results = func(query)
        end = time.time()
        if isinstance(results, list) and results and isinstance(results[0], tuple):
            top_docs = [doc_id for doc_id, _ in results[:3]]
        else:
            top_docs = list(results)[:3]
        print(f"{name}: Time {end-start:.6f}s, Top docs: {top_docs}")


Query: 'fox'
Boolean AND: Time 0.001551s, Top docs: [0, 1, 5]
TF-IDF: Time 0.000445s, Top docs: [0, 1, 5]
BM25: Time 0.000320s, Top docs: [5, 1, 0]
DAAT: Time 0.000363s, Top docs: [0, 1, 5]
TAAT: Time 0.000291s, Top docs: [0, 1, 5]
SAAT: Time 0.000304s, Top docs: [0, 1, 5]

Query: 'brown fox'
Boolean AND: Time 0.001131s, Top docs: [0, 1]
TF-IDF: Time 0.000338s, Top docs: [0, 1, 5]
BM25: Time 0.000335s, Top docs: [1, 0, 5]
DAAT: Time 0.000411s, Top docs: [0, 1, 5]
TAAT: Time 0.000386s, Top docs: [0, 1, 5]
SAAT: Time 0.000314s, Top docs: [0, 1, 5]

Query: 'animal'
Boolean AND: Time 0.000731s, Top docs: [4, 6]
TF-IDF: Time 0.000427s, Top docs: [4, 6]
BM25: Time 0.000489s, Top docs: [4, 6]
DAAT: Time 0.000659s, Top docs: [4, 6, 0]
TAAT: Time 0.000497s, Top docs: [4, 6]
SAAT: Time 0.000305s, Top docs: [4, 6]


## Retrieval Quality Comparison

Demonstrate how ranked retrieval improves over boolean by providing relevance scores and retrieving documents with partial matches.

In [22]:
# Retrieval quality
query = "quick animal"
print(f"Query: '{query}'")

# Boolean
bool_results = boolean_search_and(query.split())
print(f"Boolean AND results: {bool_results}")
for doc in bool_results:
    print(f"  Doc {doc+1}: {corpus[doc]}")

# TF-IDF
tfidf_results = tfidf_search(query)
print(f"\nTF-IDF results:")
for doc_id, score in tfidf_results[:5]:
    print(f"  Doc {doc_id+1}: Score {score:.4f} - {corpus[doc_id]}")

# BM25
bm25_results = bm25_search(query)
print(f"\nBM25 results:")
for doc_id, score in bm25_results[:5]:
    print(f"  Doc {doc_id+1}: Score {score:.4f} - {corpus[doc_id]}")

print("\nImprovement: Boolean requires exact matches, ranked retrieval finds relevant documents with partial matches and ranks by relevance.")

Query: 'quick animal'
Boolean AND results: set()

TF-IDF results:
  Doc 1: Score 1.6094 - The quick brown fox jumps over the lazy dog.
  Doc 2: Score 1.6094 - A brown fox is quick and agile.
  Doc 5: Score 1.6094 - Dogs are loyal animals.
  Doc 7: Score 1.6094 - Animals live in various habitats.

BM25 results:
  Doc 5: Score 1.7399 - Dogs are loyal animals.
  Doc 2: Score 1.5328 - A brown fox is quick and agile.
  Doc 7: Score 1.5328 - Animals live in various habitats.
  Doc 1: Score 1.2380 - The quick brown fox jumps over the lazy dog.

Improvement: Boolean requires exact matches, ranked retrieval finds relevant documents with partial matches and ranks by relevance.
